In [ ]:
!apt install swig cmake ffmpeg xvfb python3-opengl

In [ ]:
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

%env MUJOCO_GL=egl
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
# Prepare to load data from google drive
from google.colab import drive
import os
import datetime

# CONNECT TO GOOGLE DRIVE
gdrive_path = '/content/drive'
drive.mount(gdrive_path)

# DEFINE WORK DIRECTORY
current_step = 'step_003'
# workDir = f'{gdrive_path}/My Drive/Research/{current_step}'
workDir = os.path.join(gdrive_path, 'My Drive', 'Research', current_step)
print('WorkDir:', workDir)

log_dir = os.path.join(workDir, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
print('LogDir:', log_dir)

tf_log_dir = os.path.join(workDir, 'tf_logs')
print('TfLogDir:', tf_log_dir)


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
import multiprocessing
print(multiprocessing.cpu_count())
print(os.cpu_count())
print(len(os.sched_getaffinity(0)))

In [ ]:
# clean content folder
import os
import shutil

# location
location = "/content"

# directories
dirs = ["sample_data", "rl-zoo", "gym_darwin_op3", "videos"]

for dir in dirs:
    path = os.path.join(location, dir)
    try:
        shutil.rmtree(path)
    except OSError as e:
        print("Error: %s : %s" % (path, e.strerror))

In [ ]:
# Install Darwin Model
model_path = '/content/gym_darwin_op3'

if os.path.isdir(model_path):
  print(f"The directory '{model_path}' exists - git pull")
  %cd {model_path}
  !git pull
  %cd /
else:
  print(f"The directory '{model_path}' does not exist - git clone")
  !git clone --single-branch --branch {current_step} https://github.com/Gianzanti/robofei_mestrado.git {model_path}


In [ ]:
!pip install -e {model_path}

In [ ]:
# Install RL Zoo
trainner_path = '/content/rl-zoo'

if os.path.isdir(trainner_path):
  print(f"The directory '{trainner_path}' exists - git pull")
  %cd {trainner_path}
  !git pull
  %cd /
else:
  print(f"The directory '{trainner_path}' does not exist - git clone")
  !git clone https://github.com/Gianzanti/rl-zoo.git {trainner_path}


In [ ]:
!pip install -e {trainner_path}

In [ ]:
path = os.path.join(trainner_path, "logs")
if os.path.isdir(path):
  shutil.rmtree(path)

path = os.path.join(trainner_path, "research_logs/DarwinOp3-v2")
if os.path.isdir(path):
  shutil.rmtree(path)

In [ ]:
# Hyper Parameters Tunning
%cd {trainner_path}

algos_cpu = ['ppo', 'a2c']
algos_cuda = ['ddpg', 'sac', 'td3']

n_timestep = 100_000
save_freq = int(n_timestep / 4)
eval_freq = int(n_timestep / 4)

max_episode_timesteps = 1000

for algo in algos_cpu:
  print('Training:', algo)
  config = f'research_config/{algo}.yml'

  !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
    --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
    --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
    --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 \
    --hyperparams n_timesteps:{timestep} \
      env_wrapper=[{"gymnasium.wrappers.TimeLimit": {"max_episode_steps": 1000}}],\
    --device cpu
  # forward_velocity_weight:5.0 ctrl_cost_weight:1e-3
  !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n 500 \
    --load-best -o "{log_dir}" -f "{log_dir}"

for algo in algos_cuda:
  print('Training:', algo)
  config = f'research_config/{algo}.yml'

  !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
    --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
    --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
    --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 \
    --hyperparams n_timesteps:{timestep} \
      env_wrapper=[{"gymnasium.wrappers.TimeLimit": {"max_episode_steps": 1000}}],\
    --device cuda
  # forward_velocity_weight:5.0 ctrl_cost_weight:1e-3
  !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n 500 \
    --load-best -o "{log_dir}" -f "{log_dir}"

